# Data pipeline walkthrough

This notebook walks through `data_cleaning/`, the pipeline that turns AGG's historical holdings into cleaned,
feature-engineered monthly bond data from WRDS, plus a daily Treasury zero curve. The universe changes month by month: for each month, the iShares holdings
file from the last trading day of the previous month (the start-of-month portfolio) decides which bonds count, so
nothing from a later date leaks into the history. Each section runs
one step of the pipeline on real data (cached pulls in `data/raw/`, so it does not hit WRDS unless a cache is missing),
and the last section looks at the finished tables in `data/processed/`.

To (re)build everything from the command line:

```sh
uv run python -m data_cleaning.run_pipeline      # ~1 h first time for 2022-2026 (WRDS pulls), ~30 min from caches
uv run python -m data_cleaning.validate          # sanity checks
```

| step | module | source | output |
|---|---|---|---|
| 1 | `holdings.py` | BlackRock holdings API, one file per month | start-of-month portfolio per month; union of bonds to pull |
| 2 | `pull.py`, `treasury_curve.py` | CRSP `tfz_dly` / `tfz_iss` | daily Treasury bid/ask/mid, accrued, yield |
| 3 | `treasury_curve.py` | step 2 | daily NSS zero curve, zero yields at 8 key rates |
| 4 | `trace.py` | TRACE enhanced, FISD | cleaned daily corporate and MBS prices, volume, bid-ask (in memory) |
| 5 | `panels.py`, `bond_math.py` | steps 2-4 | month-end price per held month, monthly bid-ask and volume, solved yield and spread |
| 6 | `merge.py` | all | `agg_holdings_monthly`, `bond_universe`, `prices_monthly`, `yield_curve_daily` as parquet |

Only the zero curve is saved daily (the key-rate covariance needs daily curve changes). Steps 4a, 4b and 5 below show the daily cleaning on one cached bucket of trades; the pipeline does this in memory and keeps one row per bond per month.

## DataFrames to know

**Final tables** (`data/processed/*.parquet`, loaded in Step 6). These are what the model stage uses.

| df | grain | rows | what it is | key columns |
|---|---|---|---|---|
| `holdings` (`agg_holdings_monthly`) | (`month`, `cusip`) | 532K | **the benchmark**: AGG's start-of-month portfolio for each month. `model_weight` is b_t (sums to 1 each month over bonds priced that month) | `month`, `holdings_date`, `cusip`, `asset_type`, `agg_weight`, `price_status`, `model_weight` |
| `bond_universe` | one row per bond | 16K | every bond held in any month: when it was held, terms, price coverage | `cusip`, `sector`, `asset_type`, `first_held_month`, `last_held_month`, `has_prices`, `maturity_date`, `coupon_rate`, `coupon_frequency`, `day_count` |
| `prices` (`prices_monthly`) | (`cusip`, `month`) | 460K | **master price table**: one month-end row per bond per month it was held. `price_source` says whether the price was seen in the final 5 trading days, earlier in the month, carried forward or (MBS) filled from the coupon cohort | `clean_price`, `accrued_interest`, `dirty_price`, `price_source`, `is_stale`, `ytm_solved`, `spread_bps`, `bid_ask_spread_bps`, `par_volume`, `trade_count` |
| `yc` (`yield_curve_daily`) | `date` | 999 | daily NSS zero curve; also the trading calendar. Decimal, continuously compounded | `zero_6m` ... `zero_30y` (the 8 key rates, `KEY_COLS`), NSS params `beta0-3`, `tau1-2`, fit diagnostics |

**Intermediate dfs** (built step by step above, mostly to show how the final tables are made).

| df | step | what it is |
|---|---|---|
| `universe`, `membership` | 1 | union of bonds held in any month (becomes `bond_universe`) and the (month, cusip) start-of-month holdings (becomes `agg_holdings_monthly`) |
| `summary` | 1 | `membership` per month by asset type: bonds held and share of AGG's bond value |
| `tsy_all` | 2 | **all** nominal CRSP Treasuries, not just AGG's. Needed because the curve fit uses the whole market |
| `curve` | 3 | NSS fit per day from `tsy_all`. Same content as `yc`, plus the key-rate columns |
| `d_key` | 3 | daily key-rate changes in bps; the input to the covariance estimate |
| `raw` / `trades` / `step_log` | 4a | one bucket of raw TRACE reports, the cleaned trades, and the per-filter drop counts |
| `corp_daily` | 4a | one bond-day row per corporate in the demo bucket (face-weighted price, volume, bid/ask) |
| `terms` | 4a | FISD terms per corporate (coupon, maturity, frequency, day count), AGG as fallback |
| `raw_mbs` / `mbs_trades` / `mbs_daily` | 4b | same flow for one MBS bucket (note: the name `mbs_daily` is reused in Step 6 for the full table) |
| `feat` | 5 | `corp_daily` with solved `ytm_solved` and `spread_bps` over `curve` |

**Conventions.** `cusip` is the 9-character join key everywhere. `yield` is the reported value (TRACE or CRSP), while `ytm_solved` is our own semiannual yield from the dirty price. Spreads are in bps over the same-day zero curve. Zero yields and `yield` are decimals, so multiply by 100 for percent.


In [1]:
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))        # repo root, so `data_cleaning` imports
from data_cleaning import config as C, trace
from data_cleaning.pull import pull_fisd_terms, pull_treasuries
from data_cleaning.trace import clean_terms
from data_cleaning.treasury_curve import KEY_COLS, clean_treasuries, fit_curves
from data_cleaning.bond_math import solve_yield_and_spread
from data_cleaning.holdings import build_holdings_history

logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")
pd.set_option("display.width", 200, "display.max_columns", 40, "display.float_format", "{:,.4f}".format)
print("window:", C.START_DATE, "to", C.END_DATE)

window: 2022-01-01 to 2026-03-31


In [2]:
mbs_daily[]

SyntaxError: invalid syntax (1624261638.py, line 1)

## Step 1: the bond universe, month by month

AGG's holdings change every month (new issues enter, bonds under one year to maturity leave), so using today's file
for 2022 would both miss bonds that were held then and include bonds that did not exist yet. Instead,
`holdings.py` downloads one iShares holdings file per month from BlackRock's fund-document API: for month m, the file
from the last trading day of m-1. That start-of-month portfolio is the benchmark for the whole of month m.

From each file we keep the sectors the model covers (`config.SECTOR_MAP`): Treasuries, industrials, financials,
utilities and MBS pass-throughs. The other sectors (CMBS, ABS, sovereign, supranational, agency, local authority,
~5% of the fund) have no daily WRDS source we process. The file's `Weight (%)` is rounded to two decimals, which zeroes
most corporates, so `agg_weight` is recomputed from market value.

The union of all months' bonds (`universe`) is what gets pulled from WRDS; `membership` is the (month, cusip) table
that later filters the daily prices and gives the monthly benchmark weights.

In [3]:
universe, membership = build_holdings_history()
summary = membership.groupby(["month", "asset_type"]).agg(bonds=("cusip", "size"), weight=("agg_weight", "sum")).unstack()
summary.iloc[::6]

data_cleaning.holdings: AGG holdings: 51 months 2022-01 to 2026-03, 9010-11786 modelled bonds per month (93.4%-95.2% of bond value), 16076 distinct bonds


bonds                   weight                
asset_type corporate   mbs treasury corporate    mbs treasury
month                                                        
2022-01         6877  1884      249    0.2655 0.2741   0.3939
2022-07         7083  2003      257    0.2470 0.2779   0.4118
2023-01         7271  2116      262    0.2520 0.2757   0.4154
2023-07         7592  2196      274    0.2562 0.2695   0.4178
2024-01         7709  2338      285    0.2576 0.2658   0.4247
2024-07         7963  2432      289    0.2551 0.2569   0.4378
2025-01         8137  2583      288    0.2534 0.2517   0.4458
2025-07         8398  2723      289    0.2489 0.2446   0.4560
2026-01         8540  2808      295    0.2453 0.2421   0.4623

In [4]:
# no look-ahead: month m always uses the file dated in month m-1
print(membership[["month", "holdings_date"]].drop_duplicates().head(3).to_string(index=False))
print(f"\n{len(universe):,} distinct bonds held at some point;", universe.groupby("asset_type").size().to_dict())
universe[["cusip", "name", "sector", "first_held_month", "last_held_month", "months_held", "agg_maturity_date", "agg_coupon_pct"]].head(6)

  month holdings_date
2022-01    2021-12-31
2022-02    2022-01-31
2022-03    2022-02-28

16,076 distinct bonds held at some point; {'corporate': 12284, 'mbs': 3282, 'treasury': 510}


,cusip,name,sector,first_held_month,last_held_month,months_held,agg_maturity_date,agg_coupon_pct
0,928881AE1,VONTIER CORP 144A,industrial,2022-01,2022-01,1,2031-04-01,2.9500
1,166756AX4,CHEVRON USA INC,industrial,2022-01,2022-01,1,2049-10-15,4.2000
2,24422EUA5,JOHN DEERE CAPITAL CORP MTN,industrial,2022-01,2022-01,1,2023-01-06,2.7000
3,72650RBB7,PLAINS ALL AMERICAN PIPELINE LP,industrial,2022-01,2022-01,1,2023-01-31,2.8500
4,86765BAL3,ENERGY TRANSFER LP,industrial,2022-01,2022-01,1,2023-01-15,3.4500
5,89236TEL5,TOYOTA MOTOR CREDIT CORP MTN,industrial,2022-01,2022-01,1,2023-01-11,2.7000


## Step 2: CRSP Treasuries

CRSP quotes every Treasury every trading day (bid, ask, accrued interest, yield, duration). Cleaning:
TIPS and callables dropped, `clean_price` = (bid + ask) / 2, `dirty_price` = clean + CRSP accrued, and the 8-character
CRSP CUSIP gets its check digit so it joins to the AGG 9-character CUSIP. All nominal Treasuries are kept here
because the curve fit (step 3) uses the whole market, not just AGG's holdings.

In [5]:
tsy_all = clean_treasuries(pull_treasuries())
tsy_all[["cusip", "date", "security_type", "coupon_rate", "maturity_date", "bid_price", "ask_price", "clean_price",
         "accrued_interest", "dirty_price", "bid_ask_spread", "yield", "duration"]].head()

data_cleaning.wrds_pull: crsp_treasuries: 439302 rows from cache


,cusip,date,security_type,coupon_rate,maturity_date,bid_price,ask_price,clean_price,accrued_interest,dirty_price,bid_ask_spread,yield,duration
0,912796C31,2022-01-03,4,0.0000,2022-01-27,99.9967,99.9973,99.9970,0.0000,99.9970,0.0007,0.0005,0.0658
1,912796C31,2022-01-04,4,0.0000,2022-01-27,99.9968,99.9974,99.9971,0.0000,99.9971,0.0006,0.0005,0.0630
2,912796C31,2022-01-05,4,0.0000,2022-01-27,99.9972,99.9979,99.9976,0.0000,99.9976,0.0006,0.0004,0.0603
3,912796C31,2022-01-06,4,0.0000,2022-01-27,99.9971,99.9977,99.9974,0.0000,99.9974,0.0006,0.0005,0.0575
4,912796C31,2022-01-07,4,0.0000,2022-01-27,99.9972,99.9978,99.9975,0.0000,99.9975,0.0006,0.0005,0.0548


In [6]:
agg_tsy = set(universe.loc[universe["asset_type"] == "treasury", "cusip"])
found = agg_tsy & set(tsy_all["cusip"])
missing = universe[universe["cusip"].isin(agg_tsy - found)]
print(f"{len(tsy_all):,} quote rows on {tsy_all['date'].nunique()} trading days, {tsy_all['date'].min().date()} to {tsy_all['date'].max().date()}")
print(f"Treasuries held by AGG in the window found in CRSP: {len(found)} of {len(agg_tsy)}; the missing ones are first held after CRSP's last date:")
missing[["name", "cusip", "first_held_month", "agg_maturity_date", "agg_accrual_date"]].sort_values("first_held_month").head()

387,886 quote rows on 999 trading days, 2022-01-03 to 2025-12-31
Treasuries held by AGG in the window found in CRSP: 502 of 510; the missing ones are first held after CRSP's last date:


,name,cusip,first_held_month,agg_maturity_date,agg_accrual_date
12114,TREASURY NOTE (OLD),91282CPT2,2026-02,2029-01-15,2026-01-15
11351,TREASURY BOND (OTR),912810UT3,2026-03,2046-02-15,2026-02-15
11773,TREASURY NOTE (OTR),91282CQA2,2026-03,2029-02-15,2026-02-15
12125,TREASURY NOTE (OLD),91282CPW5,2026-03,2031-01-31,2026-01-31
12126,TREASURY NOTE (OTR),91282CQB0,2026-03,2028-02-29,2026-02-28


## Step 3: the Treasury zero curve

For each trading day a Nelson-Siegel-Svensson zero curve is fitted to the dirty prices of all nominal
Treasuries (duration-weighted least squares, with one robust pass that zero-weights outlier bonds). The zero yields
at the 8 key-rate tenors (6M ... 30Y, continuously compounded, decimal) are the model's curve factors; their daily
changes feed the key-rate covariance.

In [7]:
curve = fit_curves(tsy_all)
curve[["date"] + KEY_COLS + ["fit_rmse_price", "n_bonds_used", "n_bonds_quoted"]].tail()

data_cleaning.curve: fitted 999 daily curves, median price rmse 0.053


,date,zero_6m,zero_1y,zero_2y,zero_3y,zero_5y,zero_7y,zero_10y,zero_30y,fit_rmse_price,n_bonds_used,n_bonds_quoted
994,2025-12-24,0.0352,0.0350,0.0350,0.0354,0.0370,0.0391,0.0423,0.0497,0.0484,317,401
995,2025-12-26,0.0352,0.0350,0.0345,0.0349,0.0369,0.0392,0.0423,0.0512,0.0571,312,400
996,2025-12-29,0.0352,0.0348,0.0345,0.0348,0.0365,0.0387,0.0421,0.0501,0.0362,310,400
997,2025-12-30,0.0353,0.0347,0.0343,0.0347,0.0367,0.0390,0.0421,0.0514,0.0384,293,400
998,2025-12-31,0.0353,0.0348,0.0345,0.0352,0.0372,0.0395,0.0425,0.0526,0.0244,298,400


In [8]:
# month-end snapshot of the curve, in percent
me = curve.groupby(curve["date"].dt.to_period("M")).tail(1).set_index("date")[KEY_COLS] * 100
me.round(2).iloc[::3]

,zero_6m,zero_1y,zero_2y,zero_3y,zero_5y,zero_7y,zero_10y,zero_30y
date,,,,,,,,
2022-01-31,0.4500,0.7900,1.1800,1.3900,1.6200,1.7600,1.9000,2.2300
2022-04-29,1.4000,2.1000,2.7000,2.8900,2.9700,2.9700,2.9700,3.2800
2022-07-29,2.7500,2.9400,2.8800,2.7800,2.6800,2.6900,2.7800,3.2900
2022-10-31,4.3900,4.5600,4.5000,4.3900,4.2200,4.1300,4.1100,4.5500
2023-01-31,4.7300,4.6500,4.1900,3.8400,3.5700,3.5300,3.5700,3.7000
2023-04-28,4.9400,4.6000,4.0000,3.6900,3.4500,3.4100,3.4600,3.8600
2023-07-31,5.3600,5.2700,4.8400,4.4700,4.1200,4.0300,4.0200,4.0800
2023-10-31,5.4400,5.3400,5.0300,4.8500,4.8300,4.9300,5.0500,5.2800
2024-01-31,5.0700,4.6400,4.1500,3.9300,3.8000,3.8400,3.9600,4.3600


In [9]:
# daily changes of the key rates (bps): the input to the covariance estimate in Deliverable 3, Section 2.1
d_key = (curve.set_index("date")[KEY_COLS].diff() * 1e4).dropna()
d_key.describe().round(2)

,zero_6m,zero_1y,zero_2y,zero_3y,zero_5y,zero_7y,zero_10y,zero_30y
count,998.0000,998.0000,998.0000,998.0000,998.0000,998.0000,998.0000,998.0000
mean,0.3300,0.3100,0.2700,0.2500,0.2400,0.2400,0.2500,0.3100
std,3.5500,5.1000,6.9300,7.3500,7.1300,6.8300,6.5800,7.9000
min,-33.6600,-58.1000,-54.0700,-40.8900,-29.0800,-27.7600,-25.4900,-34.5200
25%,-1.3900,-2.1800,-3.1300,-3.6400,-3.9600,-4.0300,-3.8600,-4.3300
50%,0.3000,0.4700,0.4500,0.2700,0.4900,0.5800,0.4000,-0.0200
75%,1.9400,2.6600,4.0800,4.5100,4.7800,4.6200,4.5600,5.0000
max,28.2900,33.3700,30.0500,25.8400,22.5600,23.2400,22.8700,32.9600


## Step 4a: corporates from enhanced TRACE

TRACE is trade-level, so each day's price has to be built from the raw trade reports. The raw pull is cached in 64
hash buckets of CUSIPs in `data/raw/trace_corporate/`; this section cleans one bucket to show the steps:

1. cancellations/corrections (`trc_st` C/X, which carry the cancelled report's `msg_seq_nb`) and reversals (Y) are dropped
   together with the reports they refer to; corrected replacements (R) are kept
2. inter-dealer trades are reported by both dealers; the buy-side copy is dropped
3. when-issued, locked-in, special-condition and settlement > T+2 trades are dropped
4. Edwards-Harris-Piwowar price filters: within 10% of the daily and 9-day medians; no isolated 10% reversals

Then one row per bond-day: face-weighted price, volume, trade count, and the dealer bid/ask implied by customer trades.

In [10]:
chunk = sorted((C.RAW_DIR / "trace_corporate").glob(f"bucket_010_{C.WINDOW_TAG}_*.parquet"))[-1]
raw = pd.read_parquet(chunk)
print(f"{len(raw):,} raw trade reports for {raw['cusip_id'].nunique()} bonds")
print(raw["trc_st"].value_counts().to_dict())
raw[["cusip_id", "trd_exctn_dt", "trd_exctn_tm", "msg_seq_nb", "orig_msg_seq_nb", "trc_st", "rptd_pr", "entrd_vol_qt",
     "yld_pt", "rpt_side_cd", "cntra_mp_id", "stlmnt_dt"]].head(8)

2,041,838 raw trade reports for 186 bonds
{'T': 2027533, 'C': 4729, 'R': 4729, 'X': 4462, 'Y': 385}


,cusip_id,trd_exctn_dt,trd_exctn_tm,msg_seq_nb,orig_msg_seq_nb,trc_st,rptd_pr,entrd_vol_qt,yld_pt,rpt_side_cd,cntra_mp_id,stlmnt_dt
0,00108WAQ3,2022-05-16,14:55:00,0137983,<NA>,T,99.8360,"500,000.0000",<NA>,B,D,2022-05-18
1,00108WAQ3,2022-05-16,14:55:00,0144324,<NA>,T,99.8360,"3,000,000.0000",<NA>,S,D,2022-05-18
2,00108WAQ3,2022-05-16,14:55:00,0144325,<NA>,T,99.8360,"1,000,000.0000",<NA>,S,C,2022-05-18
3,00108WAQ3,2022-05-16,14:55:00,0153185,<NA>,T,99.8360,"500,000.0000",<NA>,S,C,2022-05-18
4,00108WAQ3,2022-05-16,14:55:00,0153188,<NA>,T,99.8360,"1,000,000.0000",<NA>,S,C,2022-05-18
5,00108WAQ3,2022-05-16,14:55:00,0153378,<NA>,T,99.8360,"3,000,000.0000",<NA>,S,D,2022-05-18
6,00108WAQ3,2022-05-16,14:55:00,0153379,<NA>,T,99.8360,"500,000.0000",<NA>,B,D,2022-05-18
7,00108WAQ3,2022-05-16,14:55:00,0153415,<NA>,T,99.8360,"3,000,000.0000",<NA>,S,D,2022-05-18


In [11]:
trades, steps = trace.clean_trades(raw, product="corporate")
step_log = pd.DataFrame(steps, columns=["step", "trades_left"])
step_log["dropped"] = -step_log["trades_left"].diff()
step_log

,step,trades_left,dropped
0,0 valid raw trades,2041838,NaN
1,1 cancels/corrections/reversals,2022692,"19,146.0000"
2,2 inter-dealer duplicates,1490536,"532,156.0000"
3,3 non-standard trades,1384579,"105,957.0000"
4,4a median filter,1383798,781.0000
5,4b reversal filter,1383796,2.0000


In [12]:
corp_daily = trace.daily_prices(trades)
print(f"{len(corp_daily):,} bond-days for {corp_daily['cusip'].nunique()} bonds")
corp_daily.head()

110,778 bond-days for 186 bonds


,cusip,date,clean_price,yield,price_low,price_high,par_volume,trade_count,dollar_volume,price_bid,price_ask,bid_ask_spread
0,00108WAQ3,2022-05-16,99.9378,NaN,99.8360,100.4800,"686,000,000.0000",170,"685,573,535.0000",100.1935,99.8754,0.0000
1,00108WAQ3,2022-05-17,99.6841,NaN,99.2530,100.4640,"89,511,000.0000",36,"89,228,196.0750",99.8205,99.9370,0.1165
2,00108WAQ3,2022-05-18,100.1616,0.0524,99.6850,100.3590,"12,610,000.0000",8,"12,630,374.9000",100.1646,100.3590,0.1944
3,00108WAQ3,2022-05-19,101.2452,0.0517,99.8300,101.7520,"6,385,000.0000",4,"6,464,507.9500",99.8300,101.5616,1.7316
4,00108WAQ3,2022-05-20,101.1975,0.0517,101.1960,101.2940,"10,160,000.0000",3,"10,281,670.4000",101.1990,NaN,NaN


**Terms and accrued interest.** Coupon, maturity, frequency and day count come from Mergent FISD
(`fisd.fisd_mergedissue`), with the AGG file as fallback. Accrued interest (30/360 for almost all corporates) is
computed at the trade date, the same convention as CRSP's accrued for Treasuries.

In [14]:
terms = clean_terms(pull_fisd_terms(universe.loc[universe["asset_type"] == "corporate", "cusip"].tolist()), universe)
print(terms["terms_source"].value_counts().to_dict(), "| day count:", terms["day_count"].value_counts().to_dict(),
      "| coupon type:", terms["coupon_type"].value_counts(dropna=False).to_dict())
terms[["cusip", "maturity_date", "coupon_rate", "coupon_type", "coupon_frequency", "day_count", "dated_date",
       "first_interest_date", "min_denomination", "amount_outstanding", "is_callable", "terms_source"]].head()

data_cleaning.wrds_pull: fisd_corporate_terms: 11851 rows from cache


data_cleaning.corporates: corporate terms: 12284 bonds, 11848 from FISD


{'fisd': 11848, 'agg_file': 436} | day count: {'30/360': 12274, 'ACT/360': 10} | coupon type: {'F': 10786, 'V': 1065, <NA>: 433}


,cusip,maturity_date,coupon_rate,coupon_type,coupon_frequency,day_count,dated_date,first_interest_date,min_denomination,amount_outstanding,is_callable,terms_source
0,928881AE1,2031-04-01,2.9500,F,2,30/360,2021-03-10,2021-10-01,"2,000.0000",10.0000,Y,fisd
1,166756AX4,2049-10-15,4.2000,F,2,30/360,2020-10-15,2021-04-15,"2,000.0000","210,572.0000",Y,fisd
2,24422EUA5,2023-01-06,2.7000,F,2,30/360,2018-01-08,2018-07-06,"1,000.0000",0.0000,N,fisd
3,72650RBB7,2023-01-31,2.8500,F,2,30/360,2012-12-10,2013-07-31,"2,000.0000",0.0000,Y,fisd
4,86765BAL3,2023-01-15,3.4500,F,2,30/360,2013-01-10,2013-07-15,"2,000.0000",0.0000,Y,fisd


## Step 4b: MBS pass-throughs

Same cleaning on the enhanced TRACE SPDS pool table (and the TBA table for AGG's 29 TBA positions). Two
differences: pools and TBAs settle on forward SIFMA dates, so the T+2 settlement filter does not apply, and TRACE
reports pool quantities in *original* face, which is scaled to current face with the pool factor on file.
MBS yields and spreads need a prepayment model and are left NaN.

In [15]:
raw_mbs = pd.read_parquet(sorted((C.RAW_DIR / "trace_mbs").glob(f"bucket_008_{C.WINDOW_TAG}_*.parquet"))[-1])
mbs_trades, mbs_steps = trace.clean_trades(raw_mbs, product="mbs_pool")
mbs_log = pd.DataFrame(mbs_steps, columns=["step", "trades_left"])
mbs_daily = trace.daily_prices(mbs_trades)
display(mbs_log)
mbs_daily[["cusip", "date", "clean_price", "price_low", "price_high", "par_volume", "trade_count", "pool_factor", "bid_ask_spread"]].head()

,step,trades_left
0,0 valid raw trades,32617
1,1 cancels/corrections/reversals,30777
2,2 inter-dealer duplicates,24406
3,3 non-standard trades,24241
4,4a median filter,24042
5,4b reversal filter,24039
6,5 pool factor available,24039


,cusip,date,clean_price,price_low,price_high,par_volume,trade_count,pool_factor,bid_ask_spread
0,3128MAET2,2023-01-20,97.9375,97.9375,97.9375,"6,561.5012",1,0.1810,NaN
1,3128MAET2,2023-12-01,94.5937,94.5938,94.5938,"28,008.4686",1,0.1626,NaN
2,3128MAET2,2024-01-09,95.2500,95.2500,95.2500,"149,506.6097",1,0.1601,NaN
3,3128MJZQ6,2022-01-03,103.0268,102.9737,103.0800,"28,218.0992",2,0.3527,NaN
4,3128MJZQ6,2022-01-06,103.0156,103.0156,103.0156,"7,054.5248",1,0.3527,NaN


## Step 5: yields and spreads

For every bond-day with bullet cash flows, `bond_math.py` solves two numbers from the dirty price: the semiannual
yield to maturity, and the constant spread (bps) over that day's fitted zero curve that reprices the bond. Treasuries
get both too, which is a check on the curve (their spread should be about zero).

In [16]:
from data_cleaning.bond_math import accrued_interest
tm = terms.set_index("cusip")
acc = np.full(len(corp_daily), np.nan)
for cusip, idx in corp_daily.groupby("cusip").indices.items():
    b = tm.loc[cusip]
    acc[idx] = accrued_interest(corp_daily["date"].values[idx], b["maturity_date"], b["coupon_rate"], int(b["coupon_frequency"]), b["day_count"])
corp_daily["accrued_interest"] = acc
corp_daily["dirty_price"] = corp_daily["clean_price"] + acc
feat = solve_yield_and_spread(corp_daily, terms, curve)
feat[["cusip", "date", "clean_price", "accrued_interest", "dirty_price", "yield", "ytm_solved", "spread_bps", "years_to_maturity"]].head()

data_cleaning.features: solved yields for 100.0% of rows, spreads for 93.9%


,cusip,date,clean_price,accrued_interest,dirty_price,yield,ytm_solved,spread_bps,years_to_maturity
0,00108WAQ3,2022-05-16,99.9378,0.0146,99.9524,NaN,0.0525,195.2376,29.9986
1,00108WAQ3,2022-05-17,99.6841,0.0292,99.7132,NaN,0.0527,190.0923,29.9959
2,00108WAQ3,2022-05-18,100.1616,0.0437,100.2053,0.0524,0.0524,197.8661,29.9932
3,00108WAQ3,2022-05-19,101.2452,0.0583,101.3036,0.0517,0.0517,192.9753,29.9904
4,00108WAQ3,2022-05-20,101.1975,0.0729,101.2705,0.0517,0.0517,199.6422,29.9877


In [17]:
# reported TRACE yield vs our solved yield (bps): small differences are settlement/convention effects
((feat["yield"] - feat["ytm_solved"]) * 1e4).describe().round(1)

count   98,723.0000
mean        -0.4000
std         23.7000
min       -475.1000
25%         -0.2000
50%          0.0000
75%          0.2000
max      4,744.8000
dtype: float64

## Step 6: the finished tables

`run_pipeline.py` does the above for every bucket, keeps one row per bond per month, and writes parquet tables to `data/processed/`. The rest of this
notebook reads those.

In [18]:
P = C.PROCESSED_DIR
holdings = pd.read_parquet(P / "agg_holdings_monthly.parquet")
bond_universe = pd.read_parquet(P / "bond_universe.parquet")
prices = pd.read_parquet(P / "prices_monthly.parquet")
yc = pd.read_parquet(P / "yield_curve_daily.parquet")
for name in ["agg_holdings_monthly", "bond_universe", "prices_monthly", "yield_curve_daily"]:
    path = P / f"{name}.parquet"
    df = pd.read_parquet(path)
    print(f"{name:20s} {len(df):>9,} rows x {df.shape[1]:2d} cols  {path.stat().st_size / 1e6:6.1f} MB   {', '.join(df.columns[:8])}, ...")

agg_holdings_monthly   531,821 rows x 12 cols    15.5 MB   month, holdings_date, cusip, sector, asset_type, is_tba, agg_market_value, agg_par_value, ...
bond_universe           16,076 rows x 32 cols     0.7 MB   cusip, name, sector, asset_type, agg_sector, is_tba, first_held_month, last_held_month, ...
prices_monthly         460,262 rows x 30 cols    34.4 MB   cusip, month, month_end_date, asset_type, sector, source, price_source, is_stale, ...
yield_curve_daily          999 rows x 18 cols     0.2 MB   date, zero_6m, zero_1y, zero_2y, zero_3y, zero_5y, zero_7y, zero_10y, ...
trace_cleaning_log          19 rows x  3 cols     0.0 MB   step, trades_left, group, ...


### `agg_holdings_monthly`: the benchmark per month

One row per (month, cusip) in AGG's start-of-month portfolio. `agg_weight` is the bond's share of the fund's bond
market value on `holdings_date`. `price_status` says whether we have a price for it that month (`observed` that
month, `filled` with a carried or MBS cohort price, or `none`), and `model_weight` renormalises `agg_weight` over
the bonds with a month-end price, so it sums to 1 every month: this is b_t.

The share of benchmark weight that is priced drops once a source ends (MBS after Feb 2025, Treasuries after Dec 2025),
so a backtest should use the months where every asset type is priced.

In [19]:
priced = holdings["agg_weight"].where(holdings["price_status"] != "none", 0.0)
coverage = (priced.groupby([holdings["month"].dt.year, holdings["asset_type"]]).sum()
            / holdings.groupby([holdings["month"].dt.year, "asset_type"])["agg_weight"].sum()).unstack()
print("share of benchmark weight with a price in the month, by year:")
coverage.round(3)

share of benchmark weight with a price in the month, by year:


asset_type,corporate,mbs,treasury
month,,,
2022,0.9800,0.8610,1.0000
2023,0.9870,0.9640,1.0000
2024,0.9890,0.9770,1.0000
2025,0.9910,0.1660,1.0000
2026,0.6590,0.0000,0.0000


In [20]:
holdings[holdings["month"] == "2024-06-01"].sample(6, random_state=0)

,month,holdings_date,cusip,sector,asset_type,is_tba,agg_market_value,agg_par_value,agg_weight_pct,agg_weight,price_status,model_weight
293225,2024-06-01,2024-05-31,3132E02B7,mbs_passthrough,mbs,False,"14,163,346.7600","13,756,943.0000",0.0100,0.0001,filled,0.0001
285404,2024-06-01,2024-05-31,031162DN7,industrial,corporate,False,"719,170.4700","710,000.0000",0.0000,0.0000,observed,0.0000
290440,2024-06-01,2024-05-31,693475BU8,financial,corporate,False,"10,914,207.5900","10,005,000.0000",0.0100,0.0001,observed,0.0001
293271,2024-06-01,2024-05-31,3132L8WD5,mbs_passthrough,mbs,False,"209,417.0600","225,489.0000",0.0000,0.0000,filled,0.0000
292580,2024-06-01,2024-05-31,949746RF0,financial,corporate,False,"11,002,426.9500","11,090,000.0000",0.0100,0.0001,observed,0.0001
293899,2024-06-01,2024-05-31,3140F9WA9,mbs_passthrough,mbs,False,"4,757,461.3100","5,298,408.0000",0.0000,0.0000,observed,0.0000


### `bond_universe`: one row per bond

Every bond held in any month of the window, with when it was held, its terms and how many held months have an observed price.

In [21]:
bond_universe[["cusip", "name", "sector", "first_held_month", "last_held_month", "months_held", "maturity_date", "coupon_rate",
               "coupon_frequency", "day_count", "terms_source", "has_prices", "n_price_months"]].sample(6, random_state=0)

,cusip,name,sector,first_held_month,last_held_month,months_held,maturity_date,coupon_rate,coupon_frequency,day_count,terms_source,has_prices,n_price_months
9653,76720AAQ9,RIO TINTO FINANCE (USA) PLC,industrial,2025-04-01,2026-03-01,12,2027-03-12,4.3750,2,30/360,fisd,True,11
3833,24703TAA4,DELL INTERNATIONAL LLC,industrial,2022-01-01,2022-06-01,6,2023-06-15,5.4500,2,30/360,fisd,True,6
12857,3131XQ5Z5,FHLMC 30YR UMBS MIRROR,mbs_passthrough,2022-01-01,2026-03-01,51,2045-01-01,3.5000,12,30/360,agg_file,True,2
4596,294429AQ8,EQUIFAX INC,industrial,2022-01-01,2023-12-01,24,2024-12-01,2.6000,2,30/360,fisd,True,24
9228,742718GA1,PROCTER & GAMBLE CO,industrial,2023-02-01,2026-03-01,38,2033-01-26,4.0500,2,30/360,fisd,True,37
10771,87612EAR7,TARGET CORPORATION,industrial,2022-01-01,2026-03-01,51,2037-10-15,6.5000,2,30/360,fisd,True,50


### `prices_monthly`: the master price table

One row per (cusip, month) for every month the bond was in AGG's start-of-month portfolio. The price is the last one
seen in the month: `traded_window` if it was in the final 5 trading days (a true month-end price), `traded_month` if
earlier, `carried` if the bond did not trade that month, `cohort_proxy` for MBS moved with their coupon cohort.
`is_stale` is True unless `traded_window`. Bid-ask spread and volume summarise every day of the month.

In [22]:
prices.groupby("asset_type").agg(rows=("month", "size"), bonds=("cusip", "nunique"), first=("month", "min"), last=("month", "max"),
                                 stale=("is_stale", "mean"), median_trade_days=("trade_days", "median"),
                                 spread_filled=("spread_bps", lambda s: s.notna().mean()),
                                 bid_ask_observed=("bid_ask_source", lambda s: (s == "observed").mean())).round(2)

,rows,bonds,first,last,stale,median_trade_days,spread_filled,bid_ask_observed
asset_type,,,,,,,,
corporate,383347,11679,2022-01-01,2026-02-01,0.0300,18.0000,0.9600,0.9200
mbs,63643,2601,2022-01-01,2025-02-01,0.8100,0.0000,0.0000,0.0800
treasury,13272,494,2022-01-01,2025-12-01,0.0000,21.0000,1.0000,1.0000


In [23]:
pd.crosstab(prices["asset_type"], prices["price_source"], normalize="index").round(3)

price_source,carried,cohort_proxy,traded_month,traded_window
asset_type,,,,
corporate,0.0030,0.0000,0.0250,0.9720
mbs,0.0130,0.6100,0.1860,0.1920
treasury,0.0000,0.0000,0.0000,1.0000


In [24]:
prices[prices["asset_type"] == "corporate"].sample(5, random_state=1)

,cusip,month,month_end_date,asset_type,sector,source,price_source,is_stale,price_date,valuation_date,months_since_trade,clean_price,accrued_interest,dirty_price,yield,ytm_solved,spread_bps,years_to_maturity,bid_ask_spread,bid_ask_spread_bps,bid_ask_source,par_volume,dollar_volume,trade_count,trade_days,pool_factor,pool_factor_date,is_tba,cohort,cohort_index
260959,681936BN9,2022-01-01,2022-01-31,corporate,financial,trace_enhanced,traded_window,False,2022-01-31,2022-01-31,0,94.9960,0.9569,95.9529,0.0380,0.0380,186.0143,11.2033,0.5350,56.3166,observed,"39,042,000.0000","37,264,595.0909",84.0000,19.0000,NaN,NaT,NaN,None,NaN
295252,754730AH2,2023-02-01,2023-02-28,corporate,financial,trace_enhanced,traded_window,False,2023-02-28,2023-02-28,0,74.6932,1.5312,76.2244,0.0554,0.0554,145.4329,28.0876,0.0000,0.0000,observed,"41,749,000.0000","32,760,137.9688",51.0000,16.0000,NaN,NaT,NaN,None,NaN
258605,674599CN3,2025-10-01,2025-10-31,corporate,industrial,trace_enhanced,traded_window,False,2025-10-31,2025-10-31,0,75.8410,0.5367,76.3777,0.0621,0.0621,150.9188,22.3710,0.5440,71.7236,observed,"26,479,000.0000","20,059,640.1907",60.0000,17.0000,NaN,NaT,NaN,None,NaN
5596,00287YCB3,2023-12-01,2023-12-29,corporate,industrial,trace_enhanced,traded_window,False,2023-12-29,2023-12-29,0,89.4958,0.4486,89.9445,0.0498,0.0498,75.5493,25.8973,0.2397,26.7820,observed,"997,405,000.0000","872,556,989.1937",557.0000,19.0000,NaN,NaT,NaN,None,NaN
125937,25746UCZ0,2022-01-01,2022-01-31,corporate,utility,trace_enhanced,traded_window,False,2022-01-27,2022-01-27,0,119.0240,1.6867,120.7107,0.0350,0.0351,137.0580,27.1184,0.0371,3.1206,observed,"11,306,000.0000","13,556,143.7747",12.0000,6.0000,NaN,NaT,NaN,None,NaN


In [25]:
prices.groupby("asset_type")[["clean_price", "accrued_interest", "ytm_solved", "spread_bps", "bid_ask_spread_bps", "dollar_volume"]].median().round(3)

,clean_price,accrued_interest,ytm_solved,spread_bps,bid_ask_spread_bps,dollar_volume
asset_type,,,,,,
corporate,95.4200,0.9440,0.0510,93.1890,24.8740,"27,184,931.5480"
mbs,93.1500,0.2820,NaN,NaN,18.4740,0.0000
treasury,95.7580,0.5230,0.0410,0.3830,3.9900,0.0000


### `yield_curve_daily`: the key-rate zero yields

One row per CRSP trading day (also the trading calendar). Below: the curve at a few month-ends, and the
correlation of daily key-rate changes, which is what the level/slope/curvature factor model summarises.

Caveat: daily changes in the 30Y zero are noisier than the other tenors and only weakly correlated with them. Few bonds pin down the long end of the NSS fit, so some of that is fit noise rather than market movement; worth keeping in mind when estimating the key-rate covariance (e.g. shrinkage, or a shorter long-end anchor).

In [26]:
(yc.set_index("date")[KEY_COLS] * 100).iloc[::63].round(2)

,zero_6m,zero_1y,zero_2y,zero_3y,zero_5y,zero_7y,zero_10y,zero_30y
date,,,,,,,,
2022-01-03,0.2000,0.4000,0.7700,1.0400,1.3700,1.5500,1.7200,2.1700
2022-04-04,1.1100,1.8000,2.4500,2.6200,2.5700,2.4800,2.4400,2.6000
2022-07-06,2.4200,2.8500,3.0100,3.0100,2.9900,3.0000,3.0600,3.3800
2022-10-04,3.8100,4.1400,4.1400,4.0300,3.8600,3.7600,3.7100,4.0500
2023-01-06,4.6700,4.6100,4.2000,3.9300,3.6700,3.5900,3.6000,3.8900
2023-04-06,4.6500,4.3400,3.8200,3.5600,3.3500,3.3100,3.3500,3.6900
2023-07-07,5.3300,5.2800,4.9200,4.6100,4.2900,4.1600,4.1000,4.1400
2023-10-05,5.4300,5.3100,4.9700,4.7600,4.6500,4.6900,4.8000,5.0800
2024-01-08,5.1200,4.7500,4.3000,4.0900,3.9600,3.9700,4.0500,4.4000


In [27]:
(yc.set_index("date")[KEY_COLS].diff().dropna().corr()).round(2)

,zero_6m,zero_1y,zero_2y,zero_3y,zero_5y,zero_7y,zero_10y,zero_30y
zero_6m,1.0000,0.8400,0.6800,0.6100,0.5400,0.4800,0.4200,0.2900
zero_1y,0.8400,1.0000,0.9300,0.8700,0.7900,0.7300,0.6400,0.3400
zero_2y,0.6800,0.9300,1.0000,0.9800,0.9200,0.8500,0.7600,0.4200
zero_3y,0.6100,0.8700,0.9800,1.0000,0.9700,0.9000,0.8100,0.5100
zero_5y,0.5400,0.7900,0.9200,0.9700,1.0000,0.9800,0.9100,0.5800
zero_7y,0.4800,0.7300,0.8500,0.9000,0.9800,1.0000,0.9700,0.5700
zero_10y,0.4200,0.6400,0.7600,0.8100,0.9100,0.9700,1.0000,0.5300
zero_30y,0.2900,0.3400,0.4200,0.5100,0.5800,0.5700,0.5300,1.0000


### make yield curve monthly

In [28]:
yc.set_index('date').resample('ME').last().tail(10)

,zero_6m,zero_1y,zero_2y,zero_3y,zero_5y,zero_7y,zero_10y,zero_30y,beta0,beta1,beta2,beta3,tau1,tau2,fit_rmse_price,n_bonds_used,n_bonds_quoted
date,,,,,,,,,,,,,,,,,
2025-03-31,0.0416,0.0400,0.0386,0.0385,0.0393,0.0405,0.0422,0.0492,0.0335,0.0081,0.0115,0.0568,0.3183,27.7585,0.0281,306,394
2025-04-30,0.0410,0.0384,0.0357,0.0355,0.0371,0.0392,0.0422,0.0498,0.0249,0.0169,0.0202,0.0820,0.3594,17.9072,0.0324,304,394
2025-05-31,0.0421,0.0408,0.0385,0.0381,0.0396,0.0417,0.0447,0.0516,0.0258,0.0132,0.0261,0.0847,0.4259,16.6152,0.0711,314,395
2025-06-30,0.0417,0.0395,0.0369,0.0365,0.0378,0.0398,0.0427,0.0514,0.0255,0.0166,0.0193,0.0857,0.4205,19.5558,0.0314,303,396
2025-07-31,0.0424,0.0412,0.0390,0.0384,0.0393,0.0412,0.0440,0.0514,0.0255,0.0164,0.0193,0.0846,0.5561,17.7722,0.0285,308,395
2025-08-31,0.0393,0.0380,0.0358,0.0353,0.0369,0.0394,0.0431,0.0516,0.0184,0.0204,0.0223,0.1084,0.5867,16.5171,0.0824,324,397
2025-09-30,0.0380,0.0368,0.0358,0.0359,0.0373,0.0392,0.0419,0.0506,0.0261,0.0140,-0.0000,0.0819,1.0025,20.6071,0.0288,305,398
2025-10-31,0.0376,0.0368,0.0356,0.0356,0.0370,0.0388,0.0414,0.0505,0.0281,0.0078,0.0148,0.0773,0.3997,23.7304,0.0350,310,398
2025-11-30,0.0370,0.0360,0.0346,0.0346,0.0360,0.0379,0.0406,0.0514,0.0270,0.0084,0.0158,0.0879,0.3858,27.1383,0.0400,297,400


### Quick validity checks

The same checks run in `data_cleaning/validate.py`.

In [29]:
tsy = prices[prices["asset_type"] == "treasury"]
checks = {
    "one row per (cusip, month)": not prices.duplicated(["cusip", "month"]).any(),
    "month m uses the file from month m-1 (no look-ahead)": (holdings["holdings_date"].dt.to_period("M") == holdings["month"].dt.to_period("M") - 1).all(),
    "model weights sum to 1 in every priced month": np.allclose(holdings.groupby("month")["model_weight"].sum().loc[lambda s: s > 0], 1.0),
    "prices only in held months": pd.MultiIndex.from_arrays([prices["cusip"], prices["month"]]).isin(
        pd.MultiIndex.from_arrays([holdings["cusip"], holdings["month"]])).all(),
    "price date not after month-end": (prices["price_date"] <= prices["month_end_date"]).all(),
    "dirty >= clean": (prices["dirty_price"] >= prices["clean_price"] - 1e-9).all(),
    "curve complete every day": yc[KEY_COLS].notna().all().all(),
    "Treasury spread over own curve ~ 0 (median bps)": abs(tsy["spread_bps"].median()) < 2,
}
for k, ok in checks.items():
    print("PASS " if ok else "FAIL ", k)
print("\nTreasury spread over fitted curve at month-end (bps):", tsy["spread_bps"].describe()[["mean", "50%", "std"]].round(1).to_dict())

PASS  one row per (cusip, month)
PASS  month m uses the file from month m-1 (no look-ahead)
PASS  model weights sum to 1 in every priced month
PASS  prices only in held months
PASS  price date not after month-end
PASS  dirty >= clean
PASS  curve complete every day
PASS  Treasury spread over own curve ~ 0 (median bps)

Treasury spread over fitted curve at month-end (bps): {'mean': -0.1, '50%': 0.4, 'std': 6.6}
